In [1]:
from Creador import generar_instancia_irp  
from Instancia import Instancia
from Proceso import Proceso
from Politica import PoliticaSimple, PoliticaSimpleClusterisada, RollOutSimple, RollOutCluster, MonteCarlo, MonteCarloRN, Qlearning, MonteCarlo_Fourier
from FuncionesAuxiliares import kmeans_clustering, visualizar_clusters, kmeans_clustering_sklearn
import numpy as np
from Generador_de_instancias import n_instancias, generar_instancias, semillas
import os, time, threading
import matplotlib.pyplot as plt

In [3]:
import time

def tarea_lenta(nombre):
    print(f"Iniciando {nombre}...")
    time.sleep(30)
    print(f"{nombre} terminada.")

# Ejecución secuencial
inicio = time.time()

tarea_lenta("Descarga de datos")
tarea_lenta("Procesamiento de archivo")
tarea_lenta("Consulta API")

fin = time.time()
print(f"Tiempo total: {fin - inicio:.2f} segundos")


Iniciando Descarga de datos...
Descarga de datos terminada.
Iniciando Procesamiento de archivo...
Procesamiento de archivo terminada.
Iniciando Consulta API...
Consulta API terminada.
Tiempo total: 90.00 segundos


In [4]:
import threading
import time

def tarea_lenta(nombre):
    print(f"Iniciando {nombre}...")
    time.sleep(30)
    print(f"{nombre} terminada.")

inicio = time.time()

# Creamos los hilos
hilo1 = threading.Thread(target=tarea_lenta, args=("Descarga de datos",))
hilo2 = threading.Thread(target=tarea_lenta, args=("Procesamiento de archivo",))
hilo3 = threading.Thread(target=tarea_lenta, args=("Consulta API",))

# Iniciamos los hilos
hilo1.start()
hilo2.start()
hilo3.start()

# Esperamos que todos terminen
hilo1.join()
hilo2.join()
hilo3.join()

fin = time.time()
print(f"Tiempo total: {fin - inicio:.2f} segundos")


Iniciando Descarga de datos...
Iniciando Procesamiento de archivo...
Iniciando Consulta API...
Descarga de datos terminada.Procesamiento de archivo terminada.

Consulta API terminada.
Tiempo total: 30.02 segundos


In [2]:
def ejecutar_simulacion(ruta_completa, resultados, index):
    instancia = Instancia(ruta_archivo=ruta_completa, umbral_inventario_clientes=0.2, umbral_inventario_vehiculos=0.1)
    proceso = Proceso(instancia)
    PoliticaRollout = RollOutSimple(instancia=instancia, proceso=proceso)
    
    trayectoria_rollout, _, _ = PoliticaRollout.run()
    costo_episodio = sum(i['recompensa'] for i in trayectoria_rollout)
    
    resultados[index] = costo_episodio


In [5]:
if __name__ == "__main__":
    
    carpeta_instancias = "Instancias demandas bajas 25C 200"
    ruta_completa = os.path.join(carpeta_instancias, f'instancia9.xml')
    # Lectura de la instancia y se guarda en un objeto instancia
    instancia = Instancia(ruta_archivo=ruta_completa, umbral_inventario_clientes= 0.2, umbral_inventario_vehiculos= 0.1)
    proceso = Proceso(instancia)
    PoliticaRollout = RollOutSimple(instancia= instancia, proceso= proceso)

    costos = np.array([])

    # ejecutamos 10 veces la política rollout
    for _ in range(4):
        trayectoria_rollout,_,_ = PoliticaRollout.run()
        costo_episodio =  sum(i['recompensa'] for i in trayectoria_rollout)
        costos = np.append(costos,costo_episodio)

    promedio_costo_rollout_simple = np.mean(costos)
    print(promedio_costo_rollout_simple) # 13m

208391.69717476034


In [6]:
import threading
import numpy as np
import os

if __name__ == "__main__":
    carpeta_instancias = "Instancias demandas bajas 25C 200"
    ruta_completa = os.path.join(carpeta_instancias, f'instancia9.xml')

    num_simulaciones = 4

    # Creamos una lista para guardar resultados
    resultados = [None] * num_simulaciones

    threads = []

    for i in range(num_simulaciones):
        t = threading.Thread(target=ejecutar_simulacion, args=(ruta_completa, resultados, i))
        threads.append(t)
        t.start()

    # Esperamos que todos los threads terminen
    for t in threads:
        t.join()

    costos = np.array(resultados)
    promedio_costo_rollout_simple = np.mean(costos)
    print(promedio_costo_rollout_simple)


204210.65679606845


In [ ]:
def ejecutar_simulacion(ruta_completa):
    instancia = Instancia(ruta_archivo=ruta_completa, umbral_inventario_clientes=0.2, umbral_inventario_vehiculos=0.1)
    proceso = Proceso(instancia)
    PoliticaRollout = RollOutSimple(instancia=instancia, proceso=proceso)
    
    trayectoria_rollout, _, _ = PoliticaRollout.run()
    costo_episodio = sum(i['recompensa'] for i in trayectoria_rollout)
    return costo_episodio


import multiprocessing as mp
import numpy as np
import os

if __name__ == "__main__":
    carpeta_instancias = "Instancias demandas bajas 25C 200"
    ruta_completa = os.path.join(carpeta_instancias, f'instancia9.xml')

    num_simulaciones = 10  # las que quieras

    # Creamos un Pool de procesos
    with mp.Pool(processes=mp.cpu_count()) as pool:
        # Ejecutamos varias simulaciones en paralelo
        resultados = pool.map(ejecutar_simulacion, [ruta_completa]*num_simulaciones)

    # Convertimos a numpy array si quieres
    costos = np.array(resultados)

    promedio_costo_rollout_simple = np.mean(costos)
    print(promedio_costo_rollout_simple)


In [ ]:
import multiprocessing as mp
import numpy as np
import os
import time

def ejecutar_simulacion(ruta_completa):
    instancia = Instancia(ruta_archivo=ruta_completa, umbral_inventario_clientes=0.2, umbral_inventario_vehiculos=0.1)
    proceso = Proceso(instancia)
    PoliticaRollout = RollOutSimple(instancia=instancia, proceso=proceso)
    
    trayectoria_rollout, _, _ = PoliticaRollout.run()
    costo_episodio = sum(i['recompensa'] for i in trayectoria_rollout)
    return costo_episodio

if __name__ == "__main__":
    carpeta_instancias = "Instancias demandas bajas 25C 200"
    ruta_completa = os.path.join(carpeta_instancias, f'instancia9.xml')

    num_simulaciones = 10
    num_procesos = 5  # <= Ajusta según tus cores

    start_time = time.time()

    with mp.Pool(processes=num_procesos) as pool:
        resultados = pool.map(ejecutar_simulacion, [ruta_completa]*num_simulaciones)

    costos = np.array(resultados)
    promedio_costo_rollout_simple = np.mean(costos)

    end_time = time.time()
    print(f"Promedio costo: {promedio_costo_rollout_simple}")
    print(f"Tiempo total: {end_time - start_time:.2f} segundos")
